# Pandas 80/80 — independent correctness audit

Each question runs the original solution and a separately written reference calculation. The comparison checks complete output rows (including duplicates), columns and numeric values. A PASS applies to the supplied dataset; it does not prove all possible edge cases.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Keep this notebook and the data/ folder together.
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('pandas_de_practice/data')
assert DATA_DIR.exists(), 'Cannot find data/. Open the extracted pandas_de_practice folder in VS Code.'

customers = pd.read_csv(DATA_DIR / 'customers.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions.csv')
pipeline_logs = pd.read_csv(DATA_DIR / 'pipeline_logs.csv')
cdc_changes = pd.read_csv(DATA_DIR / 'cdc_changes.csv')
events = pd.read_json(DATA_DIR / 'events.jsonl', lines=True)
events_duplicates = pd.read_json(DATA_DIR / 'events_duplicates.jsonl', lines=True)
customers_dirty = pd.read_json(DATA_DIR / 'customers_dirty.json')

print('Loaded:', {k:len(globals()[k]) for k in ['customers','orders','order_items','products','transactions','pipeline_logs','cdc_changes','events','events_duplicates','customers_dirty']})


import contextlib,io,traceback,csv
from pandas.testing import assert_frame_equal
AUDIT=[]

def verify(actual,expected,ordered=False):
    assert isinstance(actual,pd.DataFrame),f"Result is {type(actual)} instead of DataFrame"
    assert isinstance(expected,pd.DataFrame),f"Reference is {type(expected)} instead of DataFrame"
    assert set(actual.columns)==set(expected.columns),f"Columns differ: {list(actual.columns)} vs {list(expected.columns)}"
    cols=sorted(expected.columns)
    a=actual[cols].reset_index(drop=True).copy();e=expected[cols].reset_index(drop=True).copy()
    if not ordered:
        # Compare full row multisets while preserving duplicates; tolerate row ordering only.
        a=a.sort_values(cols,key=lambda s:s.astype(str),kind="stable").reset_index(drop=True)
        e=e.sort_values(cols,key=lambda s:s.astype(str),kind="stable").reset_index(drop=True)
    assert_frame_equal(a,e,check_dtype=False,check_names=False,check_exact=False,rtol=1e-7,atol=1e-7)


## Q01 — Preview records


In [ ]:
# Q01: Preview records
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile('result = orders.head(10)\n\ndisplay(result)\n','Q01_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=orders.iloc[:10].copy()','Q01_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q01',title='Preview records',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q01',title='Preview records',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q02 — Column projection


In [ ]:
# Q02: Column projection
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders[['order_id','customer_id','amount']]\n\ndisplay(result)\n",'Q02_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.loc[:,['order_id','customer_id','amount']].copy()",'Q02_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q02',title='Column projection',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q02',title='Column projection',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q03 — Filter completed


In [ ]:
# Q03: Filter completed
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.loc[orders['status'].eq('completed')].copy()\n\ndisplay(result)\n",'Q03_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=orders.loc[orders.status == "completed"].copy()','Q03_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q03',title='Filter completed',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q03',title='Filter completed',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q04 — High-value orders


In [ ]:
# Q04: High-value orders
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.loc[orders['amount'].ge(300)].sort_values('amount',ascending=False)\n\ndisplay(result)\n",'Q04_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders[orders.amount>=300].sort_values('amount',ascending=False)",'Q04_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q04',title='High-value orders',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q04',title='High-value orders',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q05 — Rename columns


In [ ]:
# Q05: Rename columns
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.rename(columns={'amount':'order_amount'})\n\ndisplay(result)\n",'Q05_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected.columns=['order_amount' if c=='amount' else c for c in expected.columns]",'Q05_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q05',title='Rename columns',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q05',title='Rename columns',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q06 — Create derived field


In [ ]:
# Q06: Create derived field
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.assign(amount_cad=orders['amount']*1.35)\n\ndisplay(result)\n",'Q06_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['amount_cad']=expected.amount*1.35",'Q06_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q06',title='Create derived field',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q06',title='Create derived field',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q07 — Distinct statuses


In [ ]:
# Q07: Distinct statuses
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = pd.DataFrame({'status': sorted(orders['status'].dropna().unique())})\n\ndisplay(result)\n",'Q07_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame({'status':np.sort(orders.status.dropna().unique())})",'Q07_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q07',title='Distinct statuses',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q07',title='Distinct statuses',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q08 — Missing-value profile


In [ ]:
# Q08: Missing-value profile
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = customers.isna().sum().rename('missing_count').reset_index().rename(columns={'index':'column'})\n\ndisplay(result)\n",'Q08_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame({'column':customers.columns,'missing_count':[customers[c].isna().sum() for c in customers]})",'Q08_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q08',title='Missing-value profile',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q08',title='Missing-value profile',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q09 — DataFrame metadata


In [ ]:
# Q09: DataFrame metadata
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = products.dtypes.astype(str).rename('dtype').reset_index().rename(columns={'index':'column'})\n\ndisplay(result)\n",'Q09_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame({'column':products.columns,'dtype':[str(products[c].dtype) for c in products]})",'Q09_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q09',title='DataFrame metadata',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q09',title='DataFrame metadata',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q10 — Conditional assignment


In [ ]:
# Q10: Conditional assignment
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.assign(value_band=np.where(orders['amount'].ge(250),'high','standard'))\n\ndisplay(result)\n",'Q10_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['value_band']=['high' if pd.notna(v) and v>=250 else 'standard' for v in orders.amount]",'Q10_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q10',title='Conditional assignment',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q10',title='Conditional assignment',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q11 — Multi-condition filter


In [ ]:
# Q11: Multi-condition filter
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.loc[orders['status'].eq('completed') & orders['amount'].ge(200)]\n\ndisplay(result)\n",'Q11_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders[(orders.status=='completed')&(orders.amount>=200)]",'Q11_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q11',title='Multi-condition filter',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q11',title='Multi-condition filter',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q12 — Membership filter


In [ ]:
# Q12: Membership filter
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = customers.loc[customers['country'].isin(['CA','US'])]\n\ndisplay(result)\n",'Q12_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers[(customers.country=='CA')|(customers.country=='US')]",'Q12_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q12',title='Membership filter',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q12',title='Membership filter',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q13 — Exclude cancelled


In [ ]:
# Q13: Exclude cancelled
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.loc[orders['status'].ne('cancelled')]\n\ndisplay(result)\n",'Q13_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders[orders.status!='cancelled']",'Q13_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q13',title='Exclude cancelled',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q13',title='Exclude cancelled',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q14 — Top 5 amounts


In [ ]:
# Q14: Top 5 amounts
try:
    _source = "result = orders.nlargest(5,'amount')\n\ndisplay(result)\n"
    def _check_q14(input_orders):
        before = input_orders.copy(deep=True)
        env = dict(globals(), orders=input_orders)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            exec(compile(_source, 'Q14_solution', 'exec'), env)
        actual = env['result']
        expected = before.sort_values('amount', ascending=False, kind='stable').iloc[:5]
        assert_frame_equal(actual, expected)
        assert_frame_equal(input_orders, before)
        return actual
    result = _check_q14(orders)
    for _fixture in [
        pd.DataFrame({'order_id': list(range(30)), 'amount': [100.0] * 30}, index=list(range(59, 29, -1))),
        pd.DataFrame({'order_id': [8, 1, 7, 2, 6, 3, 5], 'amount': [9., 8., 8., 8., 8., 8., 1.]}, index=[9, 2, 8, 3, 7, 4, 6]),
        orders.iloc[:3].copy(), orders.iloc[:0].copy()
    ]:
        _check_q14(_fixture)
    AUDIT.append(dict(question='Q14',title='Top 5 amounts',status='PASS',detail='Independent contract and targeted regressions passed'))
except Exception as exc:
    AUDIT.append(dict(question='Q14',title='Top 5 amounts',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q15 — Sort by two columns


In [ ]:
# Q15: Sort by two columns
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = customers.sort_values(['country','signup_date'])\n\ndisplay(result)\n",'Q15_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers.sort_values(['country','signup_date'])",'Q15_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q15',title='Sort by two columns',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q15',title='Sort by two columns',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q16 — Index by key


In [ ]:
# Q16: Index by key
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = customers.set_index('customer_id',drop=False)\n\ndisplay(result)\n",'Q16_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers.copy();expected.index=pd.Index(customers.customer_id,name='customer_id')",'Q16_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q16',title='Index by key',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q16',title='Index by key',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q17 — Case-insensitive match


In [ ]:
# Q17: Case-insensitive match
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = customers.loc[customers['segment'].str.casefold().eq('enterprise')]\n\ndisplay(result)\n",'Q17_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers[customers.segment.fillna('').str.lower()=='enterprise']",'Q17_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q17',title='Case-insensitive match',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q17',title='Case-insensitive match',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q18 — String contains


In [ ]:
# Q18: String contains
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = pipeline_logs.loc[pipeline_logs['message'].str.contains('error|fail',case=False,na=False,regex=True)]\n\ndisplay(result)\n",'Q18_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pipeline_logs[pipeline_logs.message.fillna('').str.lower().str.contains('error|fail',regex=True)]",'Q18_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q18',title='String contains',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q18',title='String contains',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q19 — Range filter


In [ ]:
# Q19: Range filter
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = transactions.loc[transactions['amount'].between(100,500)]\n\ndisplay(result)\n",'Q19_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=transactions[(transactions.amount>=100)&(transactions.amount<=500)]','Q19_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q19',title='Range filter',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q19',title='Range filter',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q20 — Stable sort


In [ ]:
# Q20: Stable sort
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.sort_values(['amount','order_id'],ascending=[False,True],kind='stable')\n\ndisplay(result)\n",'Q20_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.sort_values(['amount','order_id'],ascending=[False,True],kind='stable')",'Q20_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q20',title='Stable sort',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q20',title='Stable sort',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q21 — Status counts


In [ ]:
# Q21: Status counts
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.groupby('status',dropna=False).size().reset_index(name='order_count')\n\ndisplay(result)\n",'Q21_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.value_counts('status',dropna=False).rename('order_count').reset_index()",'Q21_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q21',title='Status counts',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q21',title='Status counts',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q22 — Revenue by status


In [ ]:
# Q22: Revenue by status
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.groupby('status',as_index=False).agg(revenue=('amount','sum'))\n\ndisplay(result)\n",'Q22_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.groupby('status',as_index=False)['amount'].sum().rename(columns={'amount':'revenue'})",'Q22_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q22',title='Revenue by status',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q22',title='Revenue by status',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q23 — Customer spend


In [ ]:
# Q23: Customer spend
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.groupby('customer_id',as_index=False).agg(total_spend=('amount','sum'))\n\ndisplay(result)\n",'Q23_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.groupby('customer_id',as_index=False,dropna=True)['amount'].sum().rename(columns={'amount':'total_spend'})",'Q23_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q23',title='Customer spend',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q23',title='Customer spend',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q24 — Order KPIs


In [ ]:
# Q24: Order KPIs
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.groupby('status',as_index=False).agg(order_count=('order_id','size'),avg_amount=('amount','mean'),min_amount=('amount','min'),max_amount=('amount','max'))\n\ndisplay(result)\n",'Q24_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.groupby('status',as_index=False).amount.agg(order_count='size',avg_amount='mean',min_amount='min',max_amount='max')",'Q24_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q24',title='Order KPIs',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q24',title='Order KPIs',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q25 — Segment distribution


In [ ]:
# Q25: Segment distribution
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = customers.groupby(['segment','country'],as_index=False).size().rename(columns={'size':'customers'})\n\ndisplay(result)\n",'Q25_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers.groupby(['segment','country'],as_index=False).agg(customers=('country','size'))",'Q25_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q25',title='Segment distribution',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q25',title='Segment distribution',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q26 — Completed revenue


In [ ]:
# Q26: Completed revenue
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.loc[orders['status'].eq('completed')].groupby('customer_id',as_index=False)['amount'].sum().rename(columns={'amount':'completed_revenue'})\n\ndisplay(result)\n",'Q26_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders[orders.status=='completed'].groupby('customer_id',as_index=False).amount.sum().rename(columns={'amount':'completed_revenue'})",'Q26_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q26',title='Completed revenue',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q26',title='Completed revenue',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q27 — Product units


In [ ]:
# Q27: Product units
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = order_items.groupby('product_id',as_index=False).agg(units_sold=('quantity','sum'))\n\ndisplay(result)\n",'Q27_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=order_items.groupby('product_id',as_index=False).quantity.sum().rename(columns={'quantity':'units_sold'})",'Q27_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q27',title='Product units',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q27',title='Product units',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q28 — Line-item sales


In [ ]:
# Q28: Line-item sales
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = order_items.assign(line_total=order_items['quantity']*order_items['unit_price']).groupby('product_id',as_index=False).agg(sales=('line_total','sum'))\n\ndisplay(result)\n",'Q28_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=order_items.assign(sales=order_items.quantity*order_items.unit_price).groupby('product_id',as_index=False).sales.sum()",'Q28_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q28',title='Line-item sales',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q28',title='Line-item sales',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q29 — Conditional aggregation


In [ ]:
# Q29: Conditional aggregation
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.assign(completed=orders['status'].eq('completed').astype(int),cancelled=orders['status'].eq('cancelled').astype(int)).groupby('customer_id',as_index=False)[['completed','cancelled']].sum()\n\ndisplay(result)\n",'Q29_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.pivot_table(index='customer_id',columns='status',values='order_id',aggfunc='count',fill_value=0).reset_index();expected=pd.DataFrame({'customer_id':expected.customer_id,'completed':expected.get('completed',pd.Series(0,index=expected.index)),'cancelled':expected.get('cancelled',pd.Series(0,index=expected.index))})",'Q29_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q29',title='Conditional aggregation',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q29',title='Conditional aggregation',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q30 — Pivot revenue


In [ ]:
# Q30: Pivot revenue
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.pivot_table(index='customer_id',columns='status',values='amount',aggfunc='sum',fill_value=0).reset_index()\n\ndisplay(result)\n",'Q30_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.groupby(['customer_id','status']).amount.sum().unstack(fill_value=0).reset_index()",'Q30_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q30',title='Pivot revenue',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q30',title='Pivot revenue',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q31 — Customer order join


In [ ]:
# Q31: Customer order join
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = orders.merge(customers[['customer_id','name','segment']],on='customer_id',how='left',validate='many_to_one')\n\ndisplay(result)\n",'Q31_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.join(customers.set_index('customer_id')[['name','segment']],on='customer_id')",'Q31_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q31',title='Customer order join',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q31',title='Customer order join',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q32 — Customer order counts


In [ ]:
# Q32: Customer order counts
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("counts = orders.groupby('customer_id').size().rename('order_count'); result = customers.join(counts,on='customer_id').assign(order_count=lambda x:x['order_count'].fillna(0).astype(int))\n\ndisplay(result)\n",'Q32_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers.copy();expected['order_count']=customers.customer_id.map(orders.customer_id.value_counts()).fillna(0).astype(int)",'Q32_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q32',title='Customer order counts',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q32',title='Customer order counts',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q33 — Order line enrichment


In [ ]:
# Q33: Order line enrichment
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = order_items.merge(orders[['order_id','order_date','status']],on='order_id',how='left',validate='many_to_one')\n\ndisplay(result)\n",'Q33_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=order_items.join(orders.set_index('order_id')[['order_date','status']],on='order_id')",'Q33_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q33',title='Order line enrichment',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q33',title='Order line enrichment',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q34 — Product line enrichment


In [ ]:
# Q34: Product line enrichment
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result = order_items.merge(products[['product_id','category']],on='product_id',how='left',validate='many_to_one')\n\ndisplay(result)\n",'Q34_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=order_items.join(products.set_index('product_id')[['category']],on='product_id')",'Q34_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q34',title='Product line enrichment',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q34',title='Product line enrichment',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q35 — Revenue by category


In [ ]:
# Q35: Revenue by category
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=order_items.merge(products[['product_id','category']],on='product_id',how='left'); result=x.assign(revenue=x['quantity']*x['unit_price']).groupby('category',as_index=False)['revenue'].sum()\n\ndisplay(result)\n",'Q35_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=order_items.join(products.set_index('product_id')[['category']],on='product_id');expected=expected.assign(revenue=expected.quantity*expected.unit_price).groupby('category',as_index=False).revenue.sum()",'Q35_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q35',title='Revenue by category',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q35',title='Revenue by category',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q36 — Unmatched customer keys


In [ ]:
# Q36: Unmatched customer keys
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.merge(customers[['customer_id']],on='customer_id',how='left',indicator=True);result=x.loc[x['_merge'].eq('left_only')].drop(columns='_merge')\n\ndisplay(result)\n",'Q36_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=orders[~orders.customer_id.isin(customers.customer_id)]','Q36_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q36',title='Unmatched customer keys',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q36',title='Unmatched customer keys',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q37 — Anti-join customers


In [ ]:
# Q37: Anti-join customers
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=customers.loc[~customers['customer_id'].isin(orders['customer_id'])]\n\ndisplay(result)\n",'Q37_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=customers[~customers.customer_id.isin(orders.customer_id)]','Q37_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q37',title='Anti-join customers',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q37',title='Anti-join customers',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q38 — Wide-to-long


In [ ]:
# Q38: Wide-to-long
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=orders[['order_id','status','amount']].melt(id_vars='order_id',var_name='attribute',value_name='value')\n\ndisplay(result)\n",'Q38_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.concat([orders[['order_id']].assign(attribute='status',value=orders.status),orders[['order_id']].assign(attribute='amount',value=orders.amount)],ignore_index=True)",'Q38_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q38',title='Wide-to-long',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q38',title='Wide-to-long',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q39 — Pivot count by segment


In [ ]:
# Q39: Pivot count by segment
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.merge(customers[['customer_id','segment']],on='customer_id',how='left');result=x.pivot_table(index='segment',columns='status',values='order_id',aggfunc='count',fill_value=0).reset_index()\n\ndisplay(result)\n",'Q39_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.join(customers.set_index('customer_id')[['segment']],on='customer_id').pivot_table(index='segment',columns='status',values='order_id',aggfunc='count',fill_value=0).reset_index()",'Q39_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q39',title='Pivot count by segment',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q39',title='Pivot count by segment',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q40 — Join cardinality check


In [ ]:
# Q40: Join cardinality check
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("joined=orders.merge(customers[['customer_id','name']],on='customer_id',how='left',validate='many_to_one');result=pd.DataFrame([{'before':len(orders),'after':len(joined),'difference':len(joined)-len(orders)}])\n\ndisplay(result)\n",'Q40_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame([{'before':len(orders),'after':len(orders),'difference':0}])",'Q40_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q40',title='Join cardinality check',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q40',title='Join cardinality check',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q41 — Parse order dates


In [ ]:
# Q41: Parse order dates
try:
    _source = "result=orders.assign(order_date=pd.to_datetime(orders['order_date']))\n\ndisplay(result)\nprint(result['order_date'].dtype)\n"
    def _check_q41(input_orders):
        before = input_orders.copy(deep=True)
        env = dict(globals(), orders=input_orders)
        output = io.StringIO()
        with contextlib.redirect_stdout(output), contextlib.redirect_stderr(io.StringIO()):
            exec(compile(_source, 'Q41_solution', 'exec'), env)
        actual = env['result']
        expected = before.copy()
        expected['order_date'] = pd.to_datetime(before['order_date'])
        assert pd.api.types.is_datetime64_any_dtype(actual['order_date'].dtype), 'order_date must have a datetime dtype'
        assert_frame_equal(actual, expected)
        assert_frame_equal(input_orders, before)
        assert str(actual['order_date'].dtype) in output.getvalue().splitlines(), 'Display the resulting order_date dtype'
        return actual
    result = _check_q41(orders)
    for _fixture in [
        pd.DataFrame({'order_id': [3, 1, 2], 'order_date': ['2025-02-03', '2025-01-01', '2025-02-03'], 'amount': [3., 1., 2.]}, index=[8, 4, 9]),
        orders.iloc[:0].copy()
    ]:
        _check_q41(_fixture)
    AUDIT.append(dict(question='Q41',title='Parse order dates',status='PASS',detail='Independent contract and targeted regressions passed'))
except Exception as exc:
    AUDIT.append(dict(question='Q41',title='Parse order dates',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q42 — Monthly revenue


In [ ]:
# Q42: Monthly revenue
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(month=pd.to_datetime(orders['order_date']).dt.to_period('M'));result=x.groupby('month',as_index=False)['amount'].sum()\n\ndisplay(result)\n",'Q42_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['month']=pd.to_datetime(expected.order_date).dt.to_period('M');expected=expected.groupby('month',as_index=False).amount.sum()",'Q42_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q42',title='Monthly revenue',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q42',title='Monthly revenue',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q43 — Daily transaction volume


In [ ]:
# Q43: Daily transaction volume
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=transactions.assign(day=pd.to_datetime(transactions['transaction_ts'],utc=True).dt.date);result=x.groupby('day',as_index=False).size().rename(columns={'size':'transactions'})\n\ndisplay(result)\n",'Q43_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=transactions.copy();expected['day']=pd.to_datetime(expected.transaction_ts,utc=True).dt.date;expected=expected.groupby('day',as_index=False).agg(transactions=('day','size'))",'Q43_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q43',title='Daily transaction volume',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q43',title='Daily transaction volume',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q44 — Latest order per customer


In [ ]:
# Q44: Latest order per customer
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(order_date=pd.to_datetime(orders['order_date']));result=x.sort_values(['order_date','order_id']).drop_duplicates('customer_id',keep='last')\n\ndisplay(result)\n",'Q44_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['order_date']=pd.to_datetime(expected.order_date);expected=expected.sort_values(['order_date','order_id']).groupby('customer_id',dropna=False).tail(1)",'Q44_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q44',title='Latest order per customer',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q44',title='Latest order per customer',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q45 — Running customer spend


In [ ]:
# Q45: Running customer spend
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(order_date=pd.to_datetime(orders['order_date'])).sort_values(['customer_id','order_date','order_id']);result=x.assign(running_spend=x.groupby('customer_id')['amount'].cumsum())\n\ndisplay(result)\n",'Q45_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['order_date']=pd.to_datetime(expected.order_date);expected=expected.sort_values(['customer_id','order_date','order_id']);expected['running_spend']=expected.groupby('customer_id').amount.cumsum()",'Q45_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q45',title='Running customer spend',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q45',title='Running customer spend',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q46 — Previous order amount


In [ ]:
# Q46: Previous order amount
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(order_date=pd.to_datetime(orders['order_date'])).sort_values(['customer_id','order_date','order_id']);result=x.assign(previous_amount=x.groupby('customer_id')['amount'].shift())\n\ndisplay(result)\n",'Q46_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['order_date']=pd.to_datetime(expected.order_date);expected=expected.sort_values(['customer_id','order_date','order_id']);expected['previous_amount']=expected.groupby('customer_id').amount.shift(1)",'Q46_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q46',title='Previous order amount',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q46',title='Previous order amount',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q47 — Days between orders


In [ ]:
# Q47: Days between orders
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(order_date=pd.to_datetime(orders['order_date'])).sort_values(['customer_id','order_date','order_id']);result=x.assign(days_since_previous=x.groupby('customer_id')['order_date'].diff().dt.days)\n\ndisplay(result)\n",'Q47_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['order_date']=pd.to_datetime(expected.order_date);expected=expected.sort_values(['customer_id','order_date','order_id']);expected['days_since_previous']=expected.groupby('customer_id').order_date.diff().dt.days",'Q47_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q47',title='Days between orders',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q47',title='Days between orders',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q48 — Monthly MoM growth


In [ ]:
# Q48: Monthly MoM growth
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(month=pd.to_datetime(orders['order_date']).dt.to_period('M')).groupby('month',as_index=False)['amount'].sum().sort_values('month');result=x.assign(mom_pct=x['amount'].pct_change()*100)\n\ndisplay(result)\n",'Q48_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['month']=pd.to_datetime(expected.order_date).dt.to_period('M');expected=expected.groupby('month',as_index=False).amount.sum().sort_values('month');expected['mom_pct']=expected.amount.div(expected.amount.shift()).sub(1).mul(100)",'Q48_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q48',title='Monthly MoM growth',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q48',title='Monthly MoM growth',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q49 — Seven-day rolling average


In [ ]:
# Q49: Seven-day rolling average
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.assign(day=pd.to_datetime(orders['order_date'])).groupby('day')['amount'].sum().sort_index().asfreq('D',fill_value=0);result=x.rolling(7,min_periods=1).mean().rename('rolling_avg_7d').reset_index()\n\ndisplay(result)\n",'Q49_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['day']=pd.to_datetime(expected.order_date);expected=expected.groupby('day').amount.sum().sort_index().asfreq('D',fill_value=0).rolling(7,min_periods=1).mean().rename('rolling_avg_7d').reset_index()",'Q49_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q49',title='Seven-day rolling average',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q49',title='Seven-day rolling average',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q50 — Timezone conversion


In [ ]:
# Q50: Timezone conversion
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=events.assign(event_ts_local=pd.to_datetime(events['event_ts'],utc=True).dt.tz_convert('America/Vancouver'))\n\ndisplay(result)\n",'Q50_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events.copy();expected['event_ts_local']=pd.to_datetime(expected.event_ts,utc=True).dt.tz_convert('America/Vancouver')",'Q50_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q50',title='Timezone conversion',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q50',title='Timezone conversion',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q51 — Normalize emails


In [ ]:
# Q51: Normalize emails
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=customers.assign(email_normalized=customers['email'].astype('string').str.strip().str.lower())\n\ndisplay(result)\n",'Q51_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers.copy();expected['email_normalized']=customers.email.astype('string').str.lower().str.strip()",'Q51_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q51',title='Normalize emails',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q51',title='Normalize emails',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q52 — Find duplicate emails


In [ ]:
# Q52: Find duplicate emails
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=customers_dirty.assign(email_normalized=customers_dirty['email'].astype('string').str.strip().str.lower());result=x.loc[x.duplicated('email_normalized',keep=False)].sort_values('email_normalized')\n\ndisplay(result)\n",'Q52_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers_dirty.copy();expected['email_normalized']=expected.email.astype('string').str.lower().str.strip();expected=expected[expected.email_normalized.duplicated(keep=False)].sort_values('email_normalized')",'Q52_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q52',title='Find duplicate emails',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q52',title='Find duplicate emails',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q53 — Keep latest CDC record


In [ ]:
# Q53: Keep latest CDC record
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=cdc_changes.assign(changed_at=pd.to_datetime(cdc_changes['changed_at'],utc=True));result=x.sort_values(['changed_at','change_id']).drop_duplicates('customer_id',keep='last')\n\ndisplay(result)\n",'Q53_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=cdc_changes.copy();expected['changed_at']=pd.to_datetime(expected.changed_at,utc=True);expected=expected.sort_values(['changed_at','change_id']).groupby('customer_id',dropna=False).tail(1)",'Q53_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q53',title='Keep latest CDC record',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q53',title='Keep latest CDC record',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q54 — Missing customer emails


In [ ]:
# Q54: Missing customer emails
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=customers_dirty['email'].astype('string').str.strip();result=customers_dirty.loc[x.isna()|x.eq('')]\n\ndisplay(result)\n",'Q54_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers_dirty[customers_dirty.email.astype('string').str.strip().fillna('').eq('')]",'Q54_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q54',title='Missing customer emails',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q54',title='Missing customer emails',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q55 — Validate positive amounts


In [ ]:
# Q55: Validate positive amounts
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=transactions.loc[transactions['amount'].isna()|transactions['amount'].le(0)]\n\ndisplay(result)\n",'Q55_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=transactions[~(transactions.amount>0)]','Q55_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q55',title='Validate positive amounts',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q55',title='Validate positive amounts',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q56 — Duplicate events


In [ ]:
# Q56: Duplicate events
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=events_duplicates.loc[events_duplicates.duplicated(['user_id','event_type','event_ts'],keep=False)]\n\ndisplay(result)\n",'Q56_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events_duplicates[events_duplicates.duplicated(['user_id','event_type','event_ts'],keep=False)]",'Q56_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q56',title='Duplicate events',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q56',title='Duplicate events',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q57 — Deduplicate event feed


In [ ]:
# Q57: Deduplicate event feed
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=events_duplicates.sort_values('event_id').drop_duplicates(['user_id','event_type','event_ts'],keep='last')\n\ndisplay(result)\n",'Q57_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events_duplicates.sort_values('event_id').groupby(['user_id','event_type','event_ts'],dropna=False).tail(1)",'Q57_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q57',title='Deduplicate event feed',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q57',title='Deduplicate event feed',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q58 — Invalid dates


In [ ]:
# Q58: Invalid dates
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=customers_dirty.loc[pd.to_datetime(customers_dirty['signup_date'],errors='coerce').isna()]\n\ndisplay(result)\n",'Q58_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers_dirty[pd.to_datetime(customers_dirty.signup_date,errors='coerce').isna()]",'Q58_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q58',title='Invalid dates',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q58',title='Invalid dates',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q59 — Reconciliation check


In [ ]:
# Q59: Reconciliation check
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("lines=order_items.assign(line_total=order_items['quantity']*order_items['unit_price']).groupby('order_id',as_index=False)['line_total'].sum();x=orders.merge(lines,on='order_id',how='left');result=x.loc[(x['amount']-x['line_total'].fillna(0)).abs().gt(0.01)].copy()\n\ndisplay(result)\n",'Q59_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("lines=order_items.assign(line_total=order_items.quantity*order_items.unit_price).groupby('order_id',as_index=False).line_total.sum();expected=orders.merge(lines,on='order_id',how='left');expected=expected[(expected.amount-expected.line_total.fillna(0)).abs()>0.01]",'Q59_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q59',title='Reconciliation check',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q59',title='Reconciliation check',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q60 — Schema audit


In [ ]:
# Q60: Schema audit
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=pd.DataFrame({'column':orders.columns,'dtype':orders.dtypes.astype(str).values,'missing':orders.isna().sum().values})\n\ndisplay(result)\n",'Q60_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame([{'column':c,'dtype':str(orders[c].dtype),'missing':orders[c].isna().sum()} for c in orders.columns])",'Q60_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q60',title='Schema audit',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q60',title='Schema audit',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q61 — Flatten event properties


In [ ]:
# Q61: Flatten event properties
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=events.join(pd.json_normalize(events['properties']).add_prefix('property_'))\n\ndisplay(result)\n",'Q61_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events.join(pd.json_normalize(events.properties).add_prefix('property_'))",'Q61_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q61',title='Flatten event properties',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q61',title='Flatten event properties',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q62 — Event type counts


In [ ]:
# Q62: Event type counts
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=events.groupby('event_type',as_index=False).size().rename(columns={'size':'events'})\n\ndisplay(result)\n",'Q62_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events.event_type.value_counts().rename('events').rename_axis('event_type').reset_index()",'Q62_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q62',title='Event type counts',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q62',title='Event type counts',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q63 — Distinct active users


In [ ]:
# Q63: Distinct active users
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=events.assign(day=pd.to_datetime(events['event_ts'],utc=True).dt.date);result=x.groupby('day',as_index=False)['user_id'].nunique().rename(columns={'user_id':'dau'})\n\ndisplay(result)\n",'Q63_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events.copy();expected['day']=pd.to_datetime(expected.event_ts,utc=True).dt.date;expected=expected.groupby('day',as_index=False).user_id.nunique().rename(columns={'user_id':'dau'})",'Q63_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q63',title='Distinct active users',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q63',title='Distinct active users',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q64 — First event per user


In [ ]:
# Q64: First event per user
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=events.assign(ts=pd.to_datetime(events['event_ts'],utc=True));result=x.sort_values(['ts','event_id']).drop_duplicates('user_id',keep='first')\n\ndisplay(result)\n",'Q64_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events.copy();expected['ts']=pd.to_datetime(expected.event_ts,utc=True);expected=expected.sort_values(['ts','event_id']).groupby('user_id',dropna=False).head(1)",'Q64_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q64',title='First event per user',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q64',title='First event per user',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q65 — Session lengths


In [ ]:
# Q65: Session lengths
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=events.assign(ts=pd.to_datetime(events['event_ts'],utc=True));result=x.groupby('session_id',as_index=False).agg(first=('ts','min'),last=('ts','max'),events=('event_id','size'));result['duration_seconds']=(result['last']-result['first']).dt.total_seconds()\n\ndisplay(result)\n",'Q65_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=events.copy();expected['ts']=pd.to_datetime(expected.event_ts,utc=True);expected=expected.groupby('session_id',as_index=False).agg(first=('ts','min'),last=('ts','max'),events=('event_id','size'));expected['duration_seconds']=(expected['last']-expected['first']).dt.total_seconds()",'Q65_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q65',title='Session lengths',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q65',title='Session lengths',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q66 — Pipeline error logs


In [ ]:
# Q66: Pipeline error logs
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=pipeline_logs.loc[pipeline_logs['level'].astype('string').str.upper().eq('ERROR')]\n\ndisplay(result)\n",'Q66_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pipeline_logs[pipeline_logs.level.astype('string').str.upper()=='ERROR']",'Q66_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q66',title='Pipeline error logs',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q66',title='Pipeline error logs',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q67 — Pipeline runtime percentiles


In [ ]:
# Q67: Pipeline runtime percentiles
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=pipeline_logs.groupby('pipeline',as_index=False).agg(p50_ms=('duration_ms','median'),p95_ms=('duration_ms',lambda s:s.quantile(.95)))\n\ndisplay(result)\n",'Q67_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pipeline_logs.groupby('pipeline',as_index=False).duration_ms.agg(p50_ms='median',p95_ms=lambda x:x.quantile(.95))",'Q67_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q67',title='Pipeline runtime percentiles',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q67',title='Pipeline runtime percentiles',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q68 — Error rate by pipeline


In [ ]:
# Q68: Error rate by pipeline
try:
    _source = "x=pipeline_logs.assign(is_error=pipeline_logs['level'].astype('string').str.upper().eq('ERROR').fillna(False));result=x.groupby('pipeline',as_index=False).agg(log_count=('log_id','size'),error_rate_pct=('is_error',lambda s:s.mean()*100)).astype({'error_rate_pct':'Float64'})\n\ndisplay(result)\n"
    def _check_q68(input_logs):
        before = input_logs.copy(deep=True)
        env = dict(globals(), pipeline_logs=input_logs)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            exec(compile(_source, 'Q68_solution', 'exec'), env)
        actual = env['result']
        # Independently count records: missing levels are non-errors but remain
        # in the denominator. Preserve the original grouping-key policy.
        records = []
        for pipeline, group in before.groupby('pipeline'):
            errors = sum(isinstance(level, str) and level.upper() == 'ERROR'
                         for level in group['level'])
            records.append({'pipeline': pipeline, 'log_count': len(group),
                            'error_rate_pct': 100.0 * errors / len(group)})
        expected = pd.DataFrame(records, columns=['pipeline', 'log_count', 'error_rate_pct'])
        expected = expected.astype({'pipeline': before['pipeline'].dtype,
                                    'log_count': 'int64', 'error_rate_pct': 'Float64'})
        assert list(actual.columns) == ['pipeline', 'log_count', 'error_rate_pct']
        assert_frame_equal(actual.sort_values('pipeline').reset_index(drop=True),
                           expected.sort_values('pipeline').reset_index(drop=True),
                           check_exact=False, rtol=1e-12, atol=1e-12)
        assert_frame_equal(input_logs, before)
        return actual

    result = _check_q68(pipeline_logs)
    _q68_cases = [
        ('null_level_denominator', ['a', 'a', 'a'], ['ERROR', 'INFO', None]),
        ('no_errors', ['a', 'a', 'a'], ['INFO', 'debug', None]),
        ('all_null_levels', ['a', 'a'], [None, pd.NA]),
        ('all_errors_case_insensitive', ['a', 'a'], ['error', 'ERROR']),
        ('multiple_pipelines', ['a', 'b', 'a', 'b', 'c'],
         ['ERROR', 'INFO', None, None, 'ERROR']),
        ('empty', [], [])
    ]
    for _label, _pipelines, _levels in _q68_cases:
        _fixture = pd.DataFrame({
            'log_id': pd.Series(range(len(_levels)), dtype='int64'),
            'pipeline': pd.Series(_pipelines, dtype=pipeline_logs['pipeline'].dtype),
            'level': pd.Series(_levels, dtype='string')
        })
        _actual = _check_q68(_fixture)
        if _label == 'null_level_denominator':
            assert abs(float(_actual.iloc[0]['error_rate_pct']) - 100.0 / 3) < 1e-12
        if _label == 'empty':
            assert _actual.empty and pd.api.types.is_numeric_dtype(_actual['error_rate_pct'])
    AUDIT.append(dict(question='Q68',title='Error rate by pipeline',status='PASS',
                      detail='Independent full-output comparison and 6 edge-case regressions passed'))
except Exception as exc:
    AUDIT.append(dict(question='Q68',title='Error rate by pipeline',status='FAIL',
                      detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q69 — Slow pipeline records


In [ ]:
# Q69: Slow pipeline records
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=pipeline_logs.loc[pipeline_logs['duration_ms'].gt(pipeline_logs['duration_ms'].quantile(.95))]\n\ndisplay(result)\n",'Q69_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=pipeline_logs[pipeline_logs.duration_ms>pipeline_logs.duration_ms.quantile(.95)]','Q69_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q69',title='Slow pipeline records',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q69',title='Slow pipeline records',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q70 — Event funnel summary


In [ ]:
# Q70: Event funnel summary
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=events.groupby('event_type')['user_id'].nunique().reindex(['view','cart','purchase'],fill_value=0).rename('unique_users').reset_index()\n\ndisplay(result)\n",'Q70_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame({'event_type':['view','cart','purchase'],'unique_users':[events.loc[events.event_type==v,'user_id'].nunique() for v in ['view','cart','purchase']]})",'Q70_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q70',title='Event funnel summary',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q70',title='Event funnel summary',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q71 — Incremental watermark


In [ ]:
# Q71: Incremental watermark
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=orders.loc[pd.to_datetime(orders['order_date']).gt(pd.Timestamp('2025-09-01'))]\n\ndisplay(result)\n",'Q71_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders[pd.to_datetime(orders.order_date)>pd.Timestamp('2025-09-01')]",'Q71_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q71',title='Incremental watermark',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q71',title='Incremental watermark',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q72 — Idempotent batch merge


In [ ]:
# Q72: Idempotent batch merge
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("a=orders.head(100);b=orders.head(150);result=pd.concat([a,b],ignore_index=True).drop_duplicates('order_id',keep='last')\n\ndisplay(result)\n",'Q72_solution','exec'),_env)
    result=_env.get('result')
    exec(compile('expected=orders.head(150).copy()','Q72_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=True)
    AUDIT.append(dict(question='Q72',title='Idempotent batch merge',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q72',title='Idempotent batch merge',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q73 — CDC latest state


In [ ]:
# Q73: CDC latest state
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=cdc_changes.assign(ts=pd.to_datetime(cdc_changes['changed_at'],utc=True)).sort_values(['ts','change_id']).drop_duplicates('customer_id',keep='last');result=x.loc[~x['operation'].astype('string').str.upper().isin(['DELETE','D'])]\n\ndisplay(result)\n",'Q73_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=cdc_changes.copy();expected['ts']=pd.to_datetime(expected.changed_at,utc=True);expected=expected.sort_values(['ts','change_id']).groupby('customer_id',dropna=False).tail(1);expected=expected[~expected.operation.astype('string').str.upper().isin(['DELETE','D'])]",'Q73_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q73',title='CDC latest state',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q73',title='CDC latest state',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q74 — Revenue anomaly flags


In [ ]:
# Q74: Revenue anomaly flags
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("threshold=orders['amount'].mean()+2*orders['amount'].std();result=orders.assign(is_anomaly=orders['amount'].gt(threshold))\n\ndisplay(result)\n",'Q74_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=orders.copy();expected['is_anomaly']=expected.amount>expected.amount.mean()+2*expected.amount.std()",'Q74_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q74',title='Revenue anomaly flags',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q74',title='Revenue anomaly flags',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q75 — Currency normalization


In [ ]:
# Q75: Currency normalization
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("rates={'USD':1.35,'CAD':1.0,'EUR':1.48,'GBP':1.72};result=transactions.assign(amount_cad=transactions['amount']*transactions['currency'].map(rates))\n\ndisplay(result)\n",'Q75_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=transactions.copy();expected['amount_cad']=[v*{'USD':1.35,'CAD':1.0,'EUR':1.48,'GBP':1.72}.get(c,np.nan) for v,c in zip(expected.amount,expected.currency)]",'Q75_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q75',title='Currency normalization',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q75',title='Currency normalization',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q76 — Customer lifetime value


In [ ]:
# Q76: Customer lifetime value
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=orders.loc[orders['status'].eq('completed')].groupby('customer_id').agg(completed_orders=('order_id','size'),ltv=('amount','sum'));result=customers.join(x,on='customer_id');result[['completed_orders','ltv']]=result[['completed_orders','ltv']].fillna(0)\n\ndisplay(result)\n",'Q76_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=customers.copy();x=orders[orders.status=='completed'].groupby('customer_id').agg(completed_orders=('order_id','size'),ltv=('amount','sum'));expected=expected.join(x,on='customer_id');expected[['completed_orders','ltv']]=expected[['completed_orders','ltv']].fillna(0)",'Q76_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q76',title='Customer lifetime value',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q76',title='Customer lifetime value',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q77 — Top customers by segment


In [ ]:
# Q77: Top customers by segment
try:
    _source = "x=orders.loc[orders['status'].eq('completed')].groupby('customer_id',as_index=False)['amount'].sum();y=customers[['customer_id','segment','name']].merge(x,on='customer_id',how='left').fillna({'amount':0});result=y.sort_values(['segment','amount','customer_id'],ascending=[True,False,True]).groupby('segment',dropna=False).head(3)\n\ndisplay(result)\n"
    def _check_q77(input_customers, input_orders):
        env = dict(globals(), customers=input_customers, orders=input_orders)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            exec(compile(_source, 'Q77_solution', 'exec'), env)
        actual = env['result']
        completed = input_orders[input_orders.status == 'completed']
        totals = completed.groupby('customer_id', as_index=False).amount.sum()
        expected = input_customers[['customer_id', 'segment', 'name']].merge(totals, on='customer_id', how='left').fillna({'amount': 0})
        expected = expected.sort_values(['segment', 'amount', 'customer_id'], ascending=[True, False, True], na_position='last').groupby('segment', dropna=False).head(3)
        verify(actual, expected, ordered=True)
        return actual
    result = _check_q77(customers, orders)
    _fixture_customers = pd.DataFrame({
        'customer_id': [4, 3, 2, 1, 6, 5, 7],
        'segment': ['A', 'A', 'A', 'A', None, None, 'B'],
        'name': ['four', 'three', 'two', 'one', 'six', 'five', 'seven']
    })
    _fixture_orders = pd.DataFrame({
        'customer_id': [4, 3, 2, 1, 5, 7],
        'status': ['completed'] * 5 + ['pending'],
        'amount': [10., 10., 10., 10., 20., 999.]
    })
    _fixture_result = _check_q77(_fixture_customers, _fixture_orders)
    assert _fixture_result.customer_id.tolist() == [1, 2, 3, 7, 5, 6]
    assert _fixture_result.amount.tolist() == [10., 10., 10., 0., 20., 0.]
    AUDIT.append(dict(question='Q77',title='Top customers by segment',status='PASS',detail='Independent contract and targeted regressions passed'))
except Exception as exc:
    AUDIT.append(dict(question='Q77',title='Top customers by segment',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q78 — Batch quality summary


In [ ]:
# Q78: Batch quality summary
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("result=pd.DataFrame([{'rows':len(orders),'missing_customer_ids':int(orders['customer_id'].isna().sum()),'duplicate_order_ids':int(orders.duplicated('order_id').sum()),'negative_amounts':int(orders['amount'].lt(0).sum())}])\n\ndisplay(result)\n",'Q78_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pd.DataFrame([{'rows':len(orders),'missing_customer_ids':orders.customer_id.isna().sum(),'duplicate_order_ids':orders.order_id.duplicated().sum(),'negative_amounts':(orders.amount<0).sum()}])",'Q78_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q78',title='Batch quality summary',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q78',title='Batch quality summary',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q79 — Pipeline daily SLA


In [ ]:
# Q79: Pipeline daily SLA
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("x=pipeline_logs.assign(day=pd.to_datetime(pipeline_logs['timestamp'],utc=True).dt.date,breach=pipeline_logs['duration_ms'].gt(60000));result=x.groupby(['pipeline','day'],as_index=False).agg(total=('log_id','size'),sla_breaches=('breach','sum'))\n\ndisplay(result)\n",'Q79_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("expected=pipeline_logs.copy();expected['day']=pd.to_datetime(expected.timestamp,utc=True).dt.date;expected['breach']=expected.duration_ms>60000;expected=expected.groupby(['pipeline','day'],as_index=False).agg(total=('log_id','size'),sla_breaches=('breach','sum'))",'Q79_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q79',title='Pipeline daily SLA',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q79',title='Pipeline daily SLA',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


## Q80 — Order-to-transaction reconciliation


In [ ]:
# Q80: Order-to-transaction reconciliation
_env=dict(globals())
try:
    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
        exec(compile("order_total=orders.loc[orders['status'].eq('completed'),'amount'].sum();tx_total=transactions.loc[transactions['type'].astype('string').str.lower().eq('purchase'),'amount'].sum();result=pd.DataFrame([{'completed_order_total':order_total,'purchase_transaction_total':tx_total,'difference':order_total-tx_total}])\n\ndisplay(result)\n",'Q80_solution','exec'),_env)
    result=_env.get('result')
    exec(compile("a=orders.loc[orders.status=='completed','amount'].sum();b=transactions.loc[transactions.type.astype('string').str.lower()=='purchase','amount'].sum();expected=pd.DataFrame([{'completed_order_total':a,'purchase_transaction_total':b,'difference':a-b}])",'Q80_independent_reference','exec'),_env)
    verify(result,_env['expected'],ordered=False)
    AUDIT.append(dict(question='Q80',title='Order-to-transaction reconciliation',status='PASS',detail='Full output matched independent reference on supplied dataset'))
except Exception as exc:
    AUDIT.append(dict(question='Q80',title='Order-to-transaction reconciliation',status='FAIL',detail=f'{type(exc).__name__}: {str(exc)[:500]}'))
print(AUDIT[-1]['question'],AUDIT[-1]['status'],AUDIT[-1]['detail'][:180])


In [ ]:
report=pd.DataFrame(AUDIT);display(report);report.to_csv('pandas_correctness_final_report.csv',index=False);print('PASS:',sum(report.status=='PASS'),'FAIL:',sum(report.status=='FAIL'),'TOTAL:',len(report))